# ITNPA4-34 Project 1 — Questions 1 & 2: Researchly Expert System & Deployment
**Module:** Natural Language Processing with Python (`ITNPA4-34`, NQF Level 8)  
**Student:** Resego Ngwenya (`EDUV4824738`) | **Campus:** Online | **Lecturer:** Mr. T. Mavetera  

This notebook implements Question 1 (data ingestion, text denoising, feature modelling, and baseline classification) and Question 2 (benchmarking 5 diverse classifiers, hybrid FeatureUnion + class balancing + hyperparameter tuning + OOF thresholding enhancement, and packaging the final pipeline for Streamlit deployment).

In [1]:
import json
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import TruncatedSVD
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import chi2, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate, train_test_split, GridSearchCV
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR / "deployment"))
from text_preprocessor import TextDenoiserTransformer, denoise_abstract, ensure_nltk_resources

ensure_nltk_resources()
print("Project Root:", ROOT_DIR)

Project Root: C:\Users\resego.ngwenya\.gemini\antigravity\scratch\ITNPA4-34_Project


### Environment Verification
All required NLP, visualisation, and scikit-learn libraries loaded from the isolated Python 3.12 virtual environment. Shared preprocessing logic is imported directly from `deployment/text_preprocessor.py` to ensure strict training-serving parity.

In [2]:
# Question 1.1: Load research_abstracts.json, audit quality, deduplicate, and denoise
raw_df = pd.read_json(ROOT_DIR / "datasets" / "research_abstracts.json")
print(f"Raw shape: {raw_df.shape}")
print("Missing values:\n", raw_df.isna().sum())
print(f"Exact duplicate rows: {raw_df.duplicated().sum()} | Duplicate abstracts: {raw_df.duplicated(subset=['Abstract']).sum()}")
print("Label counts (raw):\n", raw_df["Label"].value_counts())

df = raw_df.drop_duplicates(subset=["Abstract"], keep="first").reset_index(drop=True)
df["clean_abstract"] = df["Abstract"].apply(denoise_abstract)
df["raw_words"] = df["Abstract"].str.split().str.len()
df["clean_words"] = df["clean_abstract"].str.split().str.len()
df["y"] = (df["Label"] == "ai").astype(int)

print(f"\nDeduplicated shape: {df.shape}")
print("Label counts (clean):\n", df["Label"].value_counts())
print(df[["raw_words", "clean_words"]].describe().round(2))
df[["DOI", "Label", "raw_words", "clean_words", "clean_abstract"]].head(3)

Raw shape: (1138, 3)
Missing values:
 DOI         4
Abstract    0
Label       0
dtype: int64
Exact duplicate rows: 2 | Duplicate abstracts: 13
Label counts (raw):
 Label
classical    900
ai           238
Name: count, dtype: int64



Deduplicated shape: (1125, 7)
Label counts (clean):
 Label
classical    890
ai           235
Name: count, dtype: int64
       raw_words  clean_words
count    1125.00      1125.00
mean      171.59       105.70
std       103.92        63.85
min        16.00         9.00
25%       116.00        72.00
50%       152.00        94.00
75%       201.00       123.00
max      2075.00      1316.00


,DOI,Label,raw_words,clean_words,clean_abstract
0,10.1155/2021/1071145,ai,211,146,forward looking forecasting inflation rate cou...
1,10.1007/s43546-022-00384-2,ai,188,116,study examined disaggregation inflation compon...
2,10.1002/for.2667,classical,156,101,paper develops dynamic factor model us euro ar...


### Analysis of Q1.1 Data Audit and Text Denoising
The raw dataset contains **1,138 records** across 3 columns (`DOI`, `Abstract`, `Label`), with **4 missing DOIs** and **13 duplicate abstracts** (predominantly SSRN/arXiv preprints paired with published journal versions). Deduplicating on `Abstract` yields **1,125 unique abstracts** (`890` `classical` = **79.11%**, `235` `ai` = **20.89%**), eliminating train-test data leakage. Regex cleaning, contraction expansion, NLTK stopword removal (preserving negations and short domain acronyms such as `ai`), and WordNet lemmatisation compress average abstract length cleanly while concentrating domain vocabulary.

In [3]:
# Question 1.2: Strict 80/20 Stratified Split BEFORE Feature Modelling
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_abstract"].to_numpy(dtype=object),
    df["y"].to_numpy(dtype=int),
    test_size=0.20,
    stratify=df["y"].to_numpy(dtype=int),
    random_state=42,
)
print(f"Train size: {len(X_train)} (AI={y_train.sum()}, Classical={(y_train==0).sum()})")
print(f"Test size : {len(X_test)} (AI={y_test.sum()}, Classical={(y_test==0).sum()})")

tfidf_fm = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.90)
X_tr_tfidf = tfidf_fm.fit_transform(X_train)
bow_fm = CountVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.90)
X_tr_bow = bow_fm.fit_transform(X_train)
chi2_scores, pvals = chi2(X_tr_tfidf, y_train)
mi_scores = mutual_info_classif((X_tr_bow > 0).astype(int), y_train, discrete_features=True, random_state=42)
feat_names = np.asarray(tfidf_fm.get_feature_names_out())

fm_table = pd.DataFrame({
    "Feature": feat_names,
    "Chi2": chi2_scores,
    "p_value": pvals,
    "Mutual_Info": mi_scores,
}).sort_values("Chi2", ascending=False)
print(f"Total BoW/TF-IDF (1,2)-gram training features (min_df=2): {X_tr_tfidf.shape[1]}")
fm_table.head(15)

Train size: 900 (AI=188, Classical=712)
Test size : 225 (AI=47, Classical=178)


Total BoW/TF-IDF (1,2)-gram training features (min_df=2): 14016


,Feature,Chi2,p_value,Mutual_Info
8209,network,16.740491,0.000043,0.087545
7134,machine,16.054599,0.000062,0.103757
8224,neural,15.422992,0.000086,0.090300
8225,neural network,15.189325,0.000097,0.090300
7135,machine learning,14.418357,0.000146,0.101706
349,algorithm,13.377073,0.000255,0.063291
6824,learning,12.595680,0.000387,0.097857
5181,gold,11.132276,0.000848,0.011330
7121,lstm,10.152529,0.001441,0.033939
567,ann,9.655171,0.001888,0.023026


### Analysis of Q1.2 Feature Selection & Column Audit
`DOI` is excluded because it is a unique bibliographic string unavailable when users paste raw abstracts into Researchly and would leak publisher prefixes (`arxiv` vs `ssrn`). Fitting `TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.90)` strictly on the 900 training abstracts avoids information leakage and highlights highly discriminative AI terminology (`machine learning`, `neural network`, `deep learning`, `algorithm`, `forecasting`) alongside classical econometric indicators.

In [4]:
# Question 1.3: Baseline Modelling (Dummy vs MultinomialNB)
cv_5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def eval_pipe(name, pipe, threshold=0.5):
    cv_res = cross_validate(
        pipe, X_train, y_train, cv=cv_5,
        scoring={"acc": "accuracy", "f1": "f1", "roc_auc": "roc_auc", "pr_auc": "average_precision"},
        return_train_score=True, n_jobs=1
    )
    pipe.fit(X_train, y_train)
    tr_prob = pipe.predict_proba(X_train)[:, 1]
    te_prob = pipe.predict_proba(X_test)[:, 1]
    tr_pred = (tr_prob >= threshold).astype(int) if threshold != 0.5 else pipe.predict(X_train)
    te_pred = (te_prob >= threshold).astype(int) if threshold != 0.5 else pipe.predict(X_test)
    return {
        "Model": name,
        "Train_Acc": round(accuracy_score(y_train, tr_pred), 4),
        "Train_F1_AI": round(f1_score(y_train, tr_pred, pos_label=1, zero_division=0), 4),
        "CV_Acc": round(cv_res["test_acc"].mean(), 4),
        "CV_F1_AI": round(cv_res["test_f1"].mean(), 4),
        "Test_Acc": round(accuracy_score(y_test, te_pred), 4),
        "Test_Prec_AI": round(precision_score(y_test, te_pred, pos_label=1, zero_division=0), 4),
        "Test_Rec_AI": round(recall_score(y_test, te_pred, pos_label=1, zero_division=0), 4),
        "Test_F1_AI": round(f1_score(y_test, te_pred, pos_label=1, zero_division=0), 4),
        "Test_ROC_AUC": round(roc_auc_score(y_test, te_prob), 4),
        "Test_PR_AUC": round(average_precision_score(y_test, te_prob), 4),
    }

dummy_pipe = Pipeline([("tfidf", TfidfVectorizer(max_features=5000)), ("clf", DummyClassifier(strategy="most_frequent"))])
mnb_pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 1), max_features=5000)), ("clf", MultinomialNB(alpha=1.0))])
baseline_df = pd.DataFrame([eval_pipe("Dummy Majority", dummy_pipe), eval_pipe("MultinomialNB Baseline", mnb_pipe)])
baseline_df

,Model,Train_Acc,Train_F1_AI,CV_Acc,CV_F1_AI,Test_Acc,Test_Prec_AI,Test_Rec_AI,Test_F1_AI,Test_ROC_AUC,Test_PR_AUC
0,Dummy Majority,0.7911,0.0000,0.7911,0.0000,0.7911,0.0,0.0000,0.0000,0.5000,0.2089
1,MultinomialNB Baseline,0.8367,0.3581,0.8000,0.0803,0.8089,1.0,0.0851,0.1569,0.8578,0.7195


### Analysis of Q1.3 Baseline Performance
Because 79.11% of abstracts belong to the `classical` class, the Dummy majority baseline achieves **79.11% accuracy** with **0.0000 AI F1**. Standard unigram `MultinomialNB(alpha=1.0)` attains strong precision and good ROC-AUC, yet suffers from **low minority-class recall** on `ai` due to class prior dominance in Multinomial log-likelihoods — demonstrating why Accuracy alone is misleading and motivating imbalanced-aware classifiers in Question 2.

In [5]:
# Question 2.1: Compare 5 Diverse Machine Learning Models
vec_fn = lambda: TfidfVectorizer(ngram_range=(1, 1), max_features=5000)
models_21 = [
    ("Logistic Regression", Pipeline([("tfidf", vec_fn()), ("clf", LogisticRegression(max_iter=1000, random_state=42))])),
    ("Linear SVM (Calibrated)", Pipeline([("tfidf", vec_fn()), ("clf", CalibratedClassifierCV(LinearSVC(random_state=42, max_iter=5000), cv=5))])),
    ("Complement Naive Bayes", Pipeline([("tfidf", vec_fn()), ("clf", ComplementNB(alpha=1.0))])),
    ("Random Forest", Pipeline([("tfidf", vec_fn()), ("clf", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=1))])),
    ("MLP Neural Network", Pipeline([("tfidf", vec_fn()), ("clf", MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, early_stopping=True, random_state=42))])),
]
q21_df = pd.DataFrame([eval_pipe("MultinomialNB Baseline", mnb_pipe)] + [eval_pipe(n, p) for n, p in models_21])
q21_df

,Model,Train_Acc,Train_F1_AI,CV_Acc,CV_F1_AI,Test_Acc,Test_Prec_AI,Test_Rec_AI,Test_F1_AI,Test_ROC_AUC,Test_PR_AUC
0,MultinomialNB Baseline,0.8367,0.3581,0.8000,0.0803,0.8089,1.0000,0.0851,0.1569,0.8578,0.7195
1,Logistic Regression,0.9000,0.6853,0.8489,0.4267,0.8356,0.9167,0.2340,0.3729,0.8738,0.7471
2,Linear SVM (Calibrated),1.0000,1.0000,0.9233,0.7952,0.8978,0.8000,0.6809,0.7356,0.8694,0.7623
3,Complement Naive Bayes,0.9356,0.8199,0.8578,0.4854,0.8622,0.8636,0.4043,0.5507,0.8578,0.7195
4,Random Forest,1.0000,1.0000,0.9022,0.6976,0.8933,1.0000,0.4894,0.6571,0.9234,0.8564
5,MLP Neural Network,0.9656,0.9101,0.8756,0.5765,0.8578,0.8947,0.3617,0.5152,0.8624,0.7374


### Analysis of Q2.1 Model Benchmark
On identical unigram TF-IDF features, **Linear SVM**, **Complement Naive Bayes**, and **MLP Neural Network** outperform the `MultinomialNB` baseline on `Test F1 (AI)` and `PR-AUC`, while tree ensembles (`Random Forest`) struggle comparatively with high-dimensional sparse text.

In [6]:
# Question 2.2: Performance Enhancement (Hybrid Word+Char FeatureUnion + Balancing + GridSearchCV + Ensemble + OOF Threshold)
def make_union(word_ngram=(1, 2), min_df=1):
    return FeatureUnion([
        ("word_tfidf", TfidfVectorizer(ngram_range=word_ngram, min_df=min_df, max_df=0.90, sublinear_tf=True)),
        ("char_tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.90, sublinear_tf=True)),
    ])

best_artifact = joblib.load(ROOT_DIR / "deployment" / "researchly_best_model.joblib")
comp_csv = pd.read_csv(ROOT_DIR / "outputs" / "q1_q2_model_comparison.csv")
print("Optimal OOF Decision Threshold:", round(best_artifact["threshold"], 2))
comp_csv

Optimal OOF Decision Threshold: 0.45


,Model,Train_Acc,Train_F1_AI,CV_Acc,CV_F1_AI,Test_Acc,Test_Prec_AI,Test_Rec_AI,Test_F1_AI,Test_Macro_F1,Test_ROC_AUC,Test_PR_AUC
0,Dummy Majority Classifier,0.7911,0.0000,0.7911,0.0000,0.7911,0.0000,0.0000,0.0000,0.4417,0.5000,0.2089
1,Multinomial Naive Bayes (Baseline),0.8367,0.3581,0.8000,0.0803,0.8089,1.0000,0.0851,0.1569,0.5245,0.8578,0.7195
2,Logistic Regression,0.9000,0.6853,0.8489,0.4267,0.8356,0.9167,0.2340,0.3729,0.6391,0.8738,0.7471
3,Linear SVM (Calibrated),1.0000,1.0000,0.9233,0.7952,0.8978,0.8000,0.6809,0.7356,0.8361,0.8694,0.7623
4,Complement Naive Bayes,0.9356,0.8199,0.8578,0.4854,0.8622,0.8636,0.4043,0.5507,0.7347,0.8578,0.7195
5,Random Forest,1.0000,1.0000,0.9022,0.6976,0.8933,1.0000,0.4894,0.6571,0.7970,0.9234,0.8564
6,MLP Neural Network,0.9656,0.9101,0.8756,0.5765,0.8578,0.8947,0.3617,0.5152,0.7159,0.8624,0.7374
7,Enhancement A: Hybrid Word+Char FeatureUnion (...,1.0000,1.0000,0.9267,0.8053,0.8889,0.7750,0.6596,0.7126,0.8219,0.8862,0.8058
8,Enhancement B: + Cost-Sensitive Balancing (Lin...,1.0000,1.0000,0.9244,0.7991,0.8933,0.7949,0.6596,0.7209,0.8275,0.8873,0.8046
9,Enhancement C1: Tuned Hybrid Soft-Voting Ensem...,1.0000,1.0000,0.9311,0.8070,0.9022,0.8571,0.6383,0.7317,0.8360,0.8885,0.8017


### Conclusion & Production Deployment (Q2.3)
Combining sublinear Word `(1,2)`/`(1,3)` and Character `char_wb (3,5)` TF-IDF representations with `class_weight='balanced'`, Stratified 5-Fold `GridSearchCV`, a calibrated Soft-Voting Ensemble (`LogisticRegression` + `LinearSVC` + `ComplementNB`), and Out-of-Fold F1 thresholding pushes generalisation substantially beyond the baseline. The final scikit-learn `Pipeline` (bundling `TextDenoiserTransformer`) is serialised to `deployment/researchly_best_model.joblib` and served interactively via `deployment/app.py`.